# 00 Forecast Setup Check

予測評価基盤の最初の動作確認ノート。ここではモデル予測は行わず、接続済みデータ、固定分割、ローリング分割、仕様レジストリ、評価指標が期待通り動くかを確認する。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "Transport_amount_project" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_connected_parcel_data
from src.forecasting.evaluation import mae, mape, mase, rmse
from src.forecasting.forecast_specs import FORECAST_SPECS, get_forecast_spec, list_forecast_specs
from src.forecasting.splits import make_fixed_split_a, make_fixed_split_b, make_rolling_splits


DATA_PATH = PROJECT_ROOT / "data" / "processed" / "parcel_volume_connected.csv"
print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)

## 1. 接続済みデータの読み込み

`data/processed/parcel_volume_connected.csv` を読み込み、予測評価に必要な `date`, `number_parcels`, `y` が使える状態になっているか確認する。`data_loader` は日付を DatetimeIndex にするため、確認用に `date` 列を戻した表も作る。

In [ ]:
df = load_connected_parcel_data(str(DATA_PATH))
df_check = df.reset_index().rename(columns={df.index.name or "index": "date"})

required_cols = ["date", "number_parcels", "y"]
column_check = pd.DataFrame(
    {
        "column": required_cols,
        "exists": [col in df_check.columns for col in required_cols],
        "dtype": [str(df_check[col].dtype) if col in df_check.columns else None for col in required_cols],
    }
)

display(column_check)
print(f"Period: {df.index.min().date()} to {df.index.max().date()}")
print(f"Rows: {len(df):,}")
display(df_check.head())
display(df_check.tail())

## 2. 固定分割A/Bの確認

固定分割Aは COVID 前までを学習し、2020-2021を評価する。固定分割Bは接続済みデータの後半を評価する。固定分割Bがデータ不足の場合は、ここで理由を表示して後続処理では skip できるようにする。

In [ ]:
split_results = {}
split_rows = []

for split_name, maker in [("fixed_a", make_fixed_split_a), ("fixed_b", make_fixed_split_b)]:
    try:
        split = maker(df)
        split_results[split_name] = split
        split_rows.append(
            {
                "split": split_name,
                "status": "success",
                "train_start": split["train"].index.min().date(),
                "train_end": split["train"].index.max().date(),
                "train_rows": len(split["train"]),
                "test_start": split["test"].index.min().date(),
                "test_end": split["test"].index.max().date(),
                "test_rows": len(split["test"]),
                "message": "",
            }
        )
    except ValueError as exc:
        split_rows.append(
            {
                "split": split_name,
                "status": "skipped",
                "train_start": None,
                "train_end": None,
                "train_rows": 0,
                "test_start": None,
                "test_end": None,
                "test_rows": 0,
                "message": str(exc),
            }
        )

split_summary = pd.DataFrame(split_rows)
display(split_summary)

## 3. ローリング分割の確認

ローリング分割は、cutoff ごとに学習期間を伸ばし、指定 horizon のターゲット月を評価するための土台。ここでは小さな例として、2024年1月から3月の cutoff に対し、1か月先と3か月先を作る。

In [ ]:
rolling_splits = make_rolling_splits(
    df,
    start_cutoff="2024-01-01",
    end_cutoff="2024-03-01",
    step_months=1,
    horizons=[1, 3],
)

rolling_summary = pd.DataFrame(
    [
        {
            "split": item["split"],
            "cutoff": item["cutoff"].date(),
            "horizon": item["horizon"],
            "train_start": item["train_start"].date(),
            "train_end": item["train_end"].date(),
            "test_start": item["test_start"].date(),
            "test_end": item["test_end"].date(),
        }
        for item in rolling_splits
    ]
)
display(rolling_summary)

## 4. FORECAST_SPECS の確認

予測評価の仕様名は notebook や script から共通利用できるように、`src.forecasting.forecast_specs` に集約する。

In [ ]:
print("Available specs:", list_forecast_specs())
display(pd.DataFrame(FORECAST_SPECS).T)
display(get_forecast_spec("baseline_m4"))

## 5. 評価指標の簡単な確認

最後に、ダミーデータで RMSE, MAE, MAPE, MASE が計算できるかを見る。実運用では `number_parcels` の原系列スケールで評価する。

In [ ]:
y_true = np.array([100.0, 120.0, 150.0, 180.0])
y_pred = np.array([110.0, 115.0, 140.0, 190.0])
y_train = np.array([80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 100, 120, 140, 160], dtype=float)

metric_check = pd.DataFrame(
    [
        {
            "rmse": rmse(y_true, y_pred),
            "mae": mae(y_true, y_pred),
            "mape": mape(y_true, y_pred),
            "mase": mase(y_true, y_pred, y_train, seasonality=12),
        }
    ]
)
display(metric_check)